# v1 commissioning (read-only)

One section per implementation stage (S0–S9). Cells only call functions from `twobunch_deploy`; each stage saves its records under `data/` (git-ignored, copied back by hand), and the result goes in the lab-notebook commissioning page.

Nothing here writes to the machine: `tests/test_read_only.py` fails on any write call in the package or in these cells.

## S0: environment

Reads no PV. Run on the host in the ACR prod Python environment, from the repo root (Jupyter started inside tmux).

Pass:
- `import_errors` is empty;
- `code_sha256` equals the value CI printed for this PR's head commit (the `pixi run code-hash` step);
- `git_commit` is that commit;
- `epics_env` shows the host's Channel Access settings (recorded; no target value).

In [ ]:
from twobunch_deploy import session

env = session.check_env()
env

In [ ]:
session.save_record("S0", "check_env", env)

## S1: camera frames and pulse IDs

Reads one PV, `CAMR:LI20:107:RAW:ArrayData` (the DTOTR2 camera image as the driver outputs it, the same arrays the DAQ saves; `Image:ArrayData` until S2): subscribes for 30 s, then disconnects. Needs the camera on, and beam for real images. Saves the summary as JSON and, next to it (`.npz`), every frame's pulse ID and timestamps plus 10 raw frames.

Pass:
- `frames` > 0 and `latency_s.first` under about 1 s: the camera is publishing, and the frame timestamps agree with this host's clock;
- `lengths` is a single value, the camera ROI's `SizeX` × `SizeY` (278,640 = 324 × 860 for the ROI read on 10-30 at 15:26; S2 checks the geometry);
- `min` ≥ 0 and `max` ≤ 4095 (12-bit camera); `dtype` recorded;
- `pulse_id_steps` almost all 36. A 0 next to a 72, or a 35 next to a 37, is the IOC stamping a frame with a neighbouring pulse ID (8 times in 33,779 DAQ frames); any other step is a finding;
- `capture_fraction` close to 1: the callback should get every frame. Lower means frames were lost between the IOC and the notebook.

In [ ]:
from twobunch_deploy import session

summary = session.capture_test(seconds=30)
summary

## S2: frame geometry

Reads the DTOTR2 camera readbacks once (`pvs.CAMERA_SETTINGS`: ROI, orientation, exposure, gain, pixel size, the driver's reverse and output size, and where the camera array comes from and its size), then subscribes to the camera for 3 s. From each frame it cuts the deployed analysis's hand crop (`screen.HAND_CROP`, 300 × 450; at the training orientation sensor columns 396–579 and rows 270–719, padded with zeros past row 184 as the DAQ classes' crop does). The crop is taken in sensor pixels, whatever ROI the camera is set to, and oriented with the live orientation readbacks. Needs beam on DTOTR2: run the S1 section again first, with beam, then this one. Saves the summary as JSON and, next to it, the pulse IDs, the newest raw frame and every frame's hand crop (`.npz`) and the figure (`.png`).

Pass:
- no `TimeoutError`: every readback answers, so the PV names are right;
- no `ValueError`: the camera array comes from the driver (port `PS107`), isn't reversed, and is the camera ROI's size (no binning); the camera's ROI contains the hand crop; each frame is `SizeX` × `SizeY` pixels;
- `trained_shape` is [300, 450];
- `differs` lists the readbacks that aren't the training values. The four ROI readbacks may differ. Orientation, exposure (`AcquireTime_RBV`: 0.1 ms in training, 1 ms on 10-30), gain or resolution (`RESOLUTION`: 30.5 µm in training, 31.22 on 10-30) differing is a finding to understand before S3: the code follows the live orientation, but the analysis constants were set with the training values;
- the camera ROI didn't change during the 3 s: the readbacks are read once, before the capture, and a move that keeps its size wouldn't be caught. Ask the operator, or run the cell twice and compare `camera`;
- the figure's left panel shows the beam where the ops DTOTR2 display shows it, with the same handedness (compare with the display, or a screenshot of it);
- the beam lies inside the outlined hand crop.

In [ ]:
from twobunch_deploy import session

geometry = session.geometry_test(seconds=3)
geometry

## S3: background and processed frames

Reads the same PVs as S2; adds none. Each frame's hand crop goes through the deployed analysis's processing (`screen.process`, Yiheng Ye's chain with the settings of `xtcav_real_features.py`): subtract a background, replace each pixel above 1000 counts by the median of its 3 × 3 neighbourhood, then set the pixels below 35 counts to 0. Then the training's energy cut (`screen.valley`: until S4 and S5, each frame's own valley) and band statistics (`screen.band_stats`), with x scaled by the live `RESOLUTION`.

Two backgrounds, both subtracted pixel by pixel from the hand crop:
- `take_background()`: the mean of 5 s of frames with no beam on DTOTR2. Needs ops to block the beam, or a beam-off window.
- `pedestal()`: one number, the median of every hand-crop pixel over 5 s of frames with beam on. For sessions without a beam-off window, once this section shows it gives the same band statistics.

Each call saves its record under `data/<date>_S3/` (JSON, `.npz`, figure). A later session can reuse a saved background with `session.load_background(path)`, `path` being the background's `.npz` as its `path` shows it.

Run the cells in order: the first with the beam off DTOTR2, the others with beam on.

### Background, beam off

Pass:
- `frames` about 50;
- every `frame_max` near the dark noise (the 10-30 dark frames peaked at 78 counts); a frame with beam reaches hundreds of counts or more. The figure's right panel is the brightest frame and must show no beam;
- `background.mean` near the dark level (31 counts on 10-30);
- `camera` recorded: the exposure, gain and ROI the background was taken at.

In [ ]:
from twobunch_deploy import session

background = session.take_background(seconds=5)
background.record

### Pedestal, beam on

Pass:
- `background.mean` (the pedestal) within about 1 count of the beam-off background's `background.mean` above;
- `fraction_above_threshold` a few percent (2.2–2.4% in the DAQ runs). Near half or more, the beam fills too much of the crop for the median to be the dark level.

In [ ]:
pedestal = session.pedestal(seconds=5)
pedestal.record

### Processing test, beam on

Every frame of a 10 s capture, processed once with each background. The `.npz` keeps every frame's hand crop, so the processing can be redone offline.

Pass:
- `frames` about 100;
- `processing_s`: `median` and `max` well under 0.1 s, one frame at 10 Hz;
- `cut_found` recorded: the fraction of frames whose own valley gives a cut. In the DAQ runs it was 0.99 (TEST_15668), 0.57 (E300_15671) and 0.94 (E300_15673). A low value means the frames rarely show two bands;
- `intensity_median` with the beam-off background compared with the DAQ runs' summed intensity per processed frame (4.7–4.9e5 in all three). Much lower means a dimmer beam than in training, and the 35-count threshold removes more of it;
- the two backgrounds' `median` values agree. In the DAQ runs, swapping the DAQ background for a pedestal moved a step's median `x_d - x_w` by a median of 0.9 µm (TEST_15668) to 7.7 µm (E300_15673), and `frac_d` by at most 0.004 in TEST_15668 (up to 0.06–0.11 in E300 steps where the valley jumps). Frame by frame it moved `x_d - x_w` by 1.4–2 µm when the cut stayed within a row; the cut moved by more than a row in 1–5% of frames, and those frames make most of the difference. Larger differences: compare the saved crops frame by frame offline before relying on the pedestal.

In [ ]:
processing = session.processing_test([background, pedestal], seconds=10)
processing

### peek(), beam on

The newest frame through the processing, with the beam-off background. Run it a few times: one frame's own valley can lock onto another dip (S5 cuts every frame of a step at the median +V cut instead).

Pass:
- `frames` 2 or more and `age_s` under about 1 s: the frame is current;
- the figure shows two bands, the cut (cyan) in the valley between them, and the drive (orange) and witness (blue) centroids on their bands;
- `band_stats.edge_frac` small: the training's validity limit is 1%;
- `processing_s` recorded.

In [ ]:
session.peek(background)